# 모듈 06: 문서 로딩과 텍스트 분할 (Document Loaders & Text Splitters)

## 이 모듈에서 배울 것

- `Document` 객체: `page_content`와 `metadata`로 구성된 LangChain의 기본 문서 단위
- `TextLoader`: `.txt` 파일을 통째로 하나의 `Document`로 로드
- `PyPDFLoader`: PDF 파일을 페이지별로 `Document` 리스트로 로드
- `RecursiveCharacterTextSplitter`: 긴 문서를 LLM이 처리 가능한 청크로 분할
- 청크 메타데이터: 각 청크가 어느 파일, 몇 페이지에서 왔는지 추적

## 학습 목표

1. **Document 객체 이해**: `page_content`와 `metadata`의 역할을 이해하고 직접 생성·접근할 수 있다
2. **TextLoader 사용**: `.txt` 파일을 로드해서 `Document` 리스트를 얻을 수 있다
3. **RecursiveCharacterTextSplitter 사용**: `chunk_size`와 `chunk_overlap` 파라미터를 조절해 문서를 청크로 분할하고, overlap 효과를 직접 확인할 수 있다

## 전체 흐름도

### RAG 파이프라인 6단계

```
[1단계: 로드]  파일(.txt, .pdf, .csv ...)  -->  Document 객체 리스트
                TextLoader / PyPDFLoader              <-- 이 모듈

[2단계: 분할]  Document 객체 리스트  -->  청크 리스트 (더 작은 Document들)
                RecursiveCharacterTextSplitter        <-- 이 모듈

[3단계: 임베딩]  청크 리스트  -->  벡터 (숫자 배열) 리스트
                 임베딩 모델                           (모듈 07)

[4단계: 저장]   벡터 리스트  -->  벡터 저장소 (Vector Store)
                                                       (모듈 07)

[5단계: 검색]   사용자 질문  -->  관련 청크 검색
                Retriever                              (모듈 08)

[6단계: 생성]   질문 + 검색된 청크  -->  LLM  -->  최종 답변
                                                       (모듈 08)
```

이 모듈은 RAG의 시작점인 **1단계(로드)** 와 **2단계(분할)** 를 다룬다.

## Document Loader란?

**복사기 비유**

`Document Loader`는 **디지털 복사기**다.

- 원본이 `.txt` 파일이든, PDF든, 웹페이지든, CSV든 상관없이
- 모두 동일한 형식(`Document` 객체)으로 복사해낸다
- 복사본에는 내용(`page_content`)과 출처 정보(`metadata`)가 함께 찍힌다

```
langchain_intro.txt  -->  TextLoader   -->  Document(page_content='...', metadata={'source': '...'})
report.pdf           -->  PyPDFLoader  -->  Document(page_content='1페이지 내용', metadata={'source': '...', 'page': 0})
                                            Document(page_content='2페이지 내용', metadata={'source': '...', 'page': 1})
                                            ...
```

**왜 통일하는가?**  
로더 이후의 모든 처리 단계(분할 → 임베딩 → 저장 → 검색)는 항상 `Document` 객체를 기준으로 동작한다.  
파일 형식에 따라 분기 처리할 필요 없이 로더만 바꾸면 된다.

In [10]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document

# .env 파일 경로: 06_document_loaders/ 폴더 기준 상위 디렉토리
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

try:
    from langchain_community.document_loaders import TextLoader
    print('[OK] TextLoader 임포트 성공')
except ImportError:
    print('[FAIL] uv add langchain-community')

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    print('[OK] RecursiveCharacterTextSplitter 임포트 성공')
except ImportError:
    print('[FAIL] uv add langchain-text-splitters')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...
[OK] TextLoader 임포트 성공
[OK] RecursiveCharacterTextSplitter 임포트 성공


## Document 객체

**라벨 붙은 카드 비유**

`Document`는 **라벨이 붙은 카드**다.

- 카드 앞면(`page_content`): 실제 텍스트 내용
- 카드 뒷면(`metadata`): 이 카드가 어디서 왔는지에 대한 정보

```python
Document(
    page_content='LangChain은 LLM 개발 프레임워크입니다.',  # 내용
    metadata={
        'source': 'langchain_intro.txt',  # 출처 파일
        'page': 0,                         # 페이지 번호 (PDF용)
        'author': '홍길동',                # 커스텀 메타데이터
    }
)
```

메타데이터는 나중에 검색 결과에서 "이 내용이 어느 문서 몇 페이지에서 왔는지" 추적할 때 사용된다.

In [11]:
from langchain_core.documents import Document

doc = Document(
    page_content='LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.',
    metadata={'source': 'sample.txt', 'page': 0, 'author': '홍길동'}
)

print(f'page_content: {doc.page_content}')
print(f'metadata: {doc.metadata}')
print(f'source: {doc.metadata["source"]}')
print(f'문서 길이: {len(doc.page_content)}자')
print('[OK] Document 객체: page_content + metadata 접근 성공')

page_content: LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
metadata: {'source': 'sample.txt', 'page': 0, 'author': '홍길동'}
source: sample.txt
문서 길이: 38자
[OK] Document 객체: page_content + metadata 접근 성공


## TextLoader

`TextLoader`는 `.txt` 파일 하나를 통째로 읽어서 **하나의 `Document`** 로 반환한다.

```python
loader = TextLoader('파일경로.txt', encoding='utf-8')
documents = loader.load()  # Document 객체 리스트 반환

len(documents)      # -> 1  (파일 전체가 하나의 Document)
documents[0].page_content  # -> 파일 전체 내용
documents[0].metadata      # -> {'source': '파일경로.txt'}
```

**TextLoader의 특징**:
- 파일 전체 = Document 1개
- `metadata['source']`에 파일 경로가 자동으로 기록됨
- `page` 정보는 없음 (텍스트 파일은 페이지 개념이 없음)

다음 셀에서 실습용 텍스트 파일을 먼저 만들자.

In [12]:
sample_text = """LangChain 소개

LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.
2022년 Harrison Chase에 의해 시작되었으며, 빠르게 성장해 AI 개발의 표준 도구 중 하나가 되었습니다.

주요 기능

LangChain의 핵심 기능은 다음과 같습니다. 첫째, 프롬프트 관리 기능을 통해 재사용 가능한
프롬프트 템플릿을 만들고 관리할 수 있습니다. 둘째, 체인 구성을 통해 여러 컴포넌트를
파이프라인 형태로 연결할 수 있습니다. 셋째, 메모리 기능으로 대화 기록을 저장하고
이전 맥락을 유지할 수 있습니다.

활용 사례

LangChain은 챗봇, 문서 요약, 코드 생성, 데이터 분석 등 다양한 분야에서 활용됩니다.
특히 RAG(Retrieval-Augmented Generation) 아키텍처를 구현하는 데 널리 사용됩니다.
RAG는 외부 문서를 검색해 LLM의 응답 품질을 향상시키는 기법입니다.

LangChain 구성 요소

주요 구성 요소로는 Model I/O(프롬프트, LLM, 출력 파서), Retrieval(문서 로더, 텍스트 분할기,
임베딩, 벡터 저장소), Memory(대화 기록), Chains(여러 컴포넌트의 파이프라인),
그리고 Agents(자율적 도구 사용)가 있습니다.
"""

data_dir = os.path.join(notebook_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
sample_path = os.path.join(data_dir, 'langchain_intro.txt')

with open(sample_path, 'w', encoding='utf-8') as f:
    f.write(sample_text)

print(f'샘플 파일 생성: {sample_path}')
print(f'파일 크기: {len(sample_text)}자')
print('[OK] 샘플 텍스트 파일 생성 완료')

샘플 파일 생성: /Users/sonny/Desktop/Dev/10X 생산성 에이전트 군단 - 사내 챗봇부터 AI 숏츠 생성기까지 외주 수익화 올인원 패키지/LangChain/06_document_loaders/data/langchain_intro.txt
파일 크기: 649자
[OK] 샘플 텍스트 파일 생성 완료


In [13]:
loader = TextLoader(sample_path, encoding='utf-8')
documents = loader.load()

print(f'로드된 Document 수: {len(documents)}개')
print(f'첫 번째 Document 길이: {len(documents[0].page_content)}자')
print(f'메타데이터: {documents[0].metadata}')
print(f'\n내용 (앞 100자):')
print(documents[0].page_content[:100] + '...')

if len(documents) >= 1:
    print('\n[OK] TextLoader: .txt 파일 전체를 하나의 Document로 로드 성공')

로드된 Document 수: 1개
첫 번째 Document 길이: 649자
메타데이터: {'source': '/Users/sonny/Desktop/Dev/10X 생산성 에이전트 군단 - 사내 챗봇부터 AI 숏츠 생성기까지 외주 수익화 올인원 패키지/LangChain/06_document_loaders/data/langchain_intro.txt'}

내용 (앞 100자):
LangChain 소개

LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.
2022년 Harrison Chase에 의해 시작...

[OK] TextLoader: .txt 파일 전체를 하나의 Document로 로드 성공


## 왜 텍스트를 분할하는가?

**카메라 뷰파인더 비유**

LLM의 컨텍스트 창은 **카메라 뷰파인더**다.

- 뷰파인더 안에 들어오는 것만 볼 수 있다
- 거대한 파노라마(긴 문서)를 한 번에 담을 수 없다
- 여러 번에 나눠서 촬영해야 한다 (= 텍스트를 청크로 분할)

```
긴 문서 (10,000자)
┌──────────────────────────────────────────────────────────────────────────┐
│ 전체 내용이 너무 길어서 LLM 컨텍스트 창에 다 들어가지 않을 수 있음       │
└──────────────────────────────────────────────────────────────────────────┘

분할 후 (청크 여러 개, 각 300자)
┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
│ 청크 1   │  │ 청크 2   │  │ 청크 3   │  │ 청크 4   │
└──────────┘  └──────────┘  └──────────┘  └──────────┘
```

**추가 이유**: RAG에서 질문과 관련 있는 청크만 골라 LLM에게 넘긴다.  
문서 전체를 넘기는 것보다 관련 청크만 넘기는 것이 더 정확하고 비용 효율적이다.

## chunk_size와 chunk_overlap

**슬라이딩 창 다이어그램**

```
원본 텍스트:
AAAAAAAABBBBBBBBCCCCCCCCDDDDDDDD

chunk_size=12, chunk_overlap=4 로 분할:

청크 1: AAAAAAAABBBB        (12자)
청크 2:         BBBBCCCCCCCC (12자, 앞에서 4자 BBBB가 겹침)
청크 3:                 CCCCDDDDDDDD (12자, 앞에서 4자 CCCC가 겹침)
        |←── 12자 ──→|
                |← 4자 →|
                |←────── 12자 ──────→|
```

- `chunk_size`: 각 청크의 최대 문자 수
- `chunk_overlap`: 인접한 청크 사이에 공유하는 문자 수

**왜 overlap이 필요한가?**  
문장이 청크 경계에서 잘리면 의미가 끊긴다.  
overlap을 두면 경계 근처의 내용이 두 청크 모두에 포함되어 맥락이 보존된다.

`RecursiveCharacterTextSplitter`는 구분자를 계층적으로 시도한다:  
`\n\n` (단락) → `\n` (줄바꿈) → `.` (문장) → ` ` (단어) → `` (문자)  
가능하면 자연스러운 경계(단락, 문장)에서 분할한다.

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=['\n\n', '\n', '.', ' ', ''],
)
chunks = splitter.split_documents(documents)

print(f'원본 문서 길이: {len(documents[0].page_content)}자')
print(f'분할된 청크 수: {len(chunks)}개')
print(f'chunk_size: 200자 (각 청크의 최대 문자 수)')
print(f'chunk_overlap: 30자 (인접 청크 간 겹치는 문자 수)')
print(f'separators: 단락 > 줄바꿈 > 문장 > 단어 > 문자 순 시도')
print('[OK] RecursiveCharacterTextSplitter 분할 완료')

원본 문서 길이: 649자
분할된 청크 수: 4개
chunk_size: 200자 (각 청크의 최대 문자 수)
chunk_overlap: 30자 (인접 청크 간 겹치는 문자 수)
separators: 단락 > 줄바꿈 > 문장 > 단어 > 문자 순 시도
[OK] RecursiveCharacterTextSplitter 분할 완료


In [15]:
print(f'총 {len(chunks)}개 청크:')
print('-' * 60)
for i, chunk in enumerate(chunks, 1):
    preview = chunk.page_content[:50].replace('\n', ' ')
    print(f'청크 {i:2d}: {len(chunk.page_content):3d}자 | {preview}...')

lengths = [len(c.page_content) for c in chunks]
print(f'\n최소 청크 길이: {min(lengths)}자')
print(f'최대 청크 길이: {max(lengths)}자')
print(f'평균 청크 길이: {sum(lengths)/len(lengths):.1f}자')
print('[OK] 청크 시각화 완료')

총 4개 청크:
------------------------------------------------------------
청크  1: 146자 | LangChain 소개  LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션...
청크  2: 183자 | 주요 기능  LangChain의 핵심 기능은 다음과 같습니다. 첫째, 프롬프트 관리 기능을...
청크  3: 180자 | 활용 사례  LangChain은 챗봇, 문서 요약, 코드 생성, 데이터 분석 등 다양한 분...
청크  4: 164자 | LangChain 구성 요소  주요 구성 요소로는 Model I/O(프롬프트, LLM, 출...

최소 청크 길이: 146자
최대 청크 길이: 183자
평균 청크 길이: 168.2자
[OK] 청크 시각화 완료


In [16]:
if len(chunks) >= 2:
    print('--- 청크 1 마지막 50자 ---')
    print(repr(chunks[0].page_content[-50:]))
    print('\n--- 청크 2 처음 80자 ---')
    print(repr(chunks[1].page_content[:80]))

    # overlap 검증
    overlap_found = False
    for n in range(5, 60):
        tail = chunks[0].page_content[-n:].strip()
        if tail and tail in chunks[1].page_content:
            print(f'\n[OK] chunk_overlap 확인: 청크 1의 마지막 {n}자가 청크 2에도 포함됨')
            overlap_found = True
            break
    if not overlap_found:
        print('\n[확인] 구분자 경계에서 분할되어 overlap이 작을 수 있습니다.')

--- 청크 1 마지막 50자 ---
'해 시작되었으며, 빠르게 성장해 AI 개발의 표준 도구 중 하나가 되었습니다.\n\n주요 기능'

--- 청크 2 처음 80자 ---
'주요 기능\n\nLangChain의 핵심 기능은 다음과 같습니다. 첫째, 프롬프트 관리 기능을 통해 재사용 가능한\n프롬프트 템플릿을 만들고 관리할 '

[OK] chunk_overlap 확인: 청크 1의 마지막 5자가 청크 2에도 포함됨


## PyPDFLoader

`PyPDFLoader`는 PDF 파일을 **페이지별로** 읽어서 **여러 개의 `Document`** 를 반환한다.

```
TextLoader  : 파일 전체 --> Document 1개
              metadata = {'source': '파일경로.txt'}

PyPDFLoader : PDF 페이지별 --> Document 여러 개
              metadata = {'source': '파일경로.pdf', 'page': 0}  (1페이지)
              metadata = {'source': '파일경로.pdf', 'page': 1}  (2페이지)
              ...
```

**핵심 차이**: `metadata['page']` 값이 생긴다.  
나중에 검색 결과에서 "몇 페이지에서 찾았는가"를 사용자에게 알려줄 수 있다.

```python
# 사용 패턴 (TextLoader와 동일)
loader = PyPDFLoader('문서.pdf')
pages = loader.load()     # 페이지별 Document 리스트
splitter.split_documents(pages)  # 이후 분할도 동일
```

In [17]:
try:
    from langchain_community.document_loaders import PyPDFLoader
    print('[OK] PyPDFLoader 임포트 성공 (pypdf 패키지 설치됨)')
    print()
    print('PyPDFLoader 사용 패턴:')
    print('  loader = PyPDFLoader("문서.pdf")')
    print('  pages = loader.load()          # 페이지별 Document 반환')
    print('  print(len(pages))              # PDF 총 페이지 수')
    print('  print(pages[0].page_content)   # 1페이지 내용')
    print('  print(pages[0].metadata)')
    print('  # -> {"source": "문서.pdf", "page": 0}')
    print()
    print('TextLoader vs PyPDFLoader:')
    print('  TextLoader  : 파일 전체 --> 하나의 Document (metadata에 page 없음)')
    print('  PyPDFLoader : 페이지별  --> 여러 Document (metadata["page"]에 페이지 번호)')
except ImportError:
    print('[FAIL] PyPDFLoader를 사용하려면: uv add pypdf')

[OK] PyPDFLoader 임포트 성공 (pypdf 패키지 설치됨)

PyPDFLoader 사용 패턴:
  loader = PyPDFLoader("문서.pdf")
  pages = loader.load()          # 페이지별 Document 반환
  print(len(pages))              # PDF 총 페이지 수
  print(pages[0].page_content)   # 1페이지 내용
  print(pages[0].metadata)
  # -> {"source": "문서.pdf", "page": 0}

TextLoader vs PyPDFLoader:
  TextLoader  : 파일 전체 --> 하나의 Document (metadata에 page 없음)
  PyPDFLoader : 페이지별  --> 여러 Document (metadata["page"]에 페이지 번호)


In [18]:
print('=== 전체 파이프라인: 로드 --> 분할 --> 청크 확인 ===')

# 1. 로드
loader_final = TextLoader(sample_path, encoding='utf-8')
docs_final = loader_final.load()
total_chars = sum(len(d.page_content) for d in docs_final)
print(f'1단계 [로드]   : {len(docs_final)}개 Document, 총 {total_chars}자')

# 2. 분할
splitter_final = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks_final = splitter_final.split_documents(docs_final)
print(f'2단계 [분할]   : {len(chunks_final)}개 청크 (chunk_size=300, chunk_overlap=50)')

# 3. 메타데이터 확인
print(f'3단계 [메타데이터]:')
for i, chunk in enumerate(chunks_final[:3], 1):
    src = os.path.basename(chunk.metadata.get('source', 'N/A'))
    print(f'  청크 {i}: source={src}, 길이={len(chunk.page_content)}자')
if len(chunks_final) > 3:
    print(f'  ... 외 {len(chunks_final)-3}개 청크')

print()
print('[OK] 전체 파이프라인 실행 완료')
print('다음 단계: 이 청크들을 임베딩 --> 벡터 저장소에 저장 (모듈 07)')

=== 전체 파이프라인: 로드 --> 분할 --> 청크 확인 ===
1단계 [로드]   : 1개 Document, 총 649자
2단계 [분할]   : 4개 청크 (chunk_size=300, chunk_overlap=50)
3단계 [메타데이터]:
  청크 1: source=langchain_intro.txt, 길이=146자
  청크 2: source=langchain_intro.txt, 길이=183자
  청크 3: source=langchain_intro.txt, 길이=180자
  ... 외 1개 청크

[OK] 전체 파이프라인 실행 완료
다음 단계: 이 청크들을 임베딩 --> 벡터 저장소에 저장 (모듈 07)


## 배운 것 정리

### 핵심 개념

**Document Loader**: 다양한 파일 형식을 `Document` 객체(page_content + metadata)로 통일해서 로드하는 컴포넌트  
**Text Splitter**: 긴 Document를 LLM 컨텍스트 창에 맞는 작은 청크로 분할하는 컴포넌트

### 핵심 클래스 요약

| 클래스 | 역할 | 출력 |
|--------|------|------|
| `Document` | 텍스트 + 메타데이터 컨테이너 | - |
| `TextLoader` | .txt 파일 로드 | Document 1개 |
| `PyPDFLoader` | PDF 파일 로드 (페이지별) | Document N개 |
| `RecursiveCharacterTextSplitter` | Document 리스트를 청크로 분할 | 청크 리스트 |

### 핵심 코드 패턴

**TextLoader 패턴**
```python
from langchain_community.document_loaders import TextLoader

loader = TextLoader('파일.txt', encoding='utf-8')
documents = loader.load()  # [Document(page_content='...', metadata={'source': '...'})]
```

**RecursiveCharacterTextSplitter 패턴**
```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,    # 청크 최대 문자 수
    chunk_overlap=50,  # 인접 청크 겹침 문자 수
)
chunks = splitter.split_documents(documents)  # Document 리스트 -> 청크 리스트
```

**전체 패턴**
```python
loader = TextLoader('파일.txt', encoding='utf-8')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
# chunks: RAG 파이프라인의 다음 단계(임베딩)로 넘어갈 준비 완료
```

### 기억할 한 줄

> **로더는 파일을 Document로 복사하고, 분할기는 Document를 청크로 잘라낸다. 이 두 단계가 RAG의 시작이다.**

## 다음 모듈 예고: 모듈 07 - 임베딩과 벡터 저장소

이 모듈에서 만든 청크들은 아직 텍스트 상태다. LLM이 관련 청크를 빠르게 찾으려면  
청크를 **숫자 벡터**로 변환해서 벡터 저장소에 넣어야 한다.

```
청크 (텍스트)  -->  임베딩 모델  -->  벡터 (숫자 배열)
                                        ↓
                                   벡터 저장소 (FAISS, Chroma ...)
                                        ↓
사용자 질문  -->  임베딩  -->  유사한 벡터 검색  -->  관련 청크 반환
```

모듈 07에서는 `GoogleGenerativeAIEmbeddings`로 청크를 임베딩하고,  
`FAISS` 벡터 저장소에 저장·검색하는 방법을 배운다.

### 준비 체크리스트

- [ ] 모듈 06 모든 셀 오류 없이 실행 완료
- [ ] 셀 5 (환경 준비): `[OK]` 3개 출력 확인 (dotenv, TextLoader, RecursiveCharacterTextSplitter)
- [ ] 셀 7 (Document 객체): `page_content`, `metadata` 접근 성공
- [ ] 셀 9 (샘플 파일 생성): `data/langchain_intro.txt` 파일 생성 확인
- [ ] 셀 10 (TextLoader): `len(documents) == 1` 및 `metadata['source']` 포함 확인
- [ ] 셀 13 (분할): 청크 수 > 1, 각 청크 길이 <= 200자 확인
- [ ] 셀 17 (PyPDFLoader): `[OK] PyPDFLoader 임포트 성공` 출력
- [ ] 셀 18 (전체 파이프라인): 3단계 순서대로 출력 완료

---
*모듈 06 완료. 다음: 모듈 07 - 임베딩과 벡터 저장소*